# Example: Static forward free-boundary equilibrium calculations using JAX

---

This example notebook shows how to use the JAX solver in FreeGSNKE to solve **static forward** free-boundary Grad-Shafranov (GS) problems.

In the **forward** solve mode we solve for the plasma equilibrium using user-defined active poloidal field coil currents, passive structure currents, and plasma current density profiles. 

Below, we illustrate how to use the solver for both diverted and limited plasma configurations in a **MAST-U-like tokamak** using stored pickle files containing the machine description. These machine description files partially come from the FreeGS repository and are not an exact replica of MAST-U. 

##### Note:
It is recommended to go through the inverse solver notebook and the default forward solver notebook before this one as we omit many of the commonly shared details!

We'll now go through the steps required to solve the **forward** problem in FreeGSNKE using JAX. 

### Initial set-up

Initially, we import modules and create the machine object and instantiate the equilibrium as normally done in FreeGSNKE.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.1, Rmax=2.0,   # radial range
    Zmin=-2.2, Zmax=2.2,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)

### Initialise JAX

To initialise the JAX solver, we import the JAX and JAX Numpy libraries. An important point to remember is that, by default, JAX performs numerical operations in single-precision arithmetic (32-bit). For many applications, this is sufficient. In scientific computing, however, double-precision arithmetic (64-bit) is considered the de-facto standard. To enable this, the JAX configuration needs to be updated and we show one way to do this. 

In [ ]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

### Instantiate limiter and current profile objects 

We need to explicitly instantiate the objects defining the limiter geometry and the toroidal current profile. The limiter geometry is taken from the equilibrium object that has already been created. There are several choices for the toroidal current profile defined in the jaxify.jtor module. 

Here, we will use the `ConstrainPaxisIp` profile. The profile parameters here match those for the Diverted case from the previous example.

In [ ]:
# initialise the limiter geometry
from freegsnke.jaxify import limiter_func
jLimiter = limiter_func.Limiter_handler(eq, eq.tokamak.limiter)

# initialise the profiles
from freegsnke.jaxify.jtor import JConstrainPaxisIp, JLao85
jProfile = JConstrainPaxisIp(
    paxis=8e3,    # profile object
    Ip=6e5,       # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

### Load the static nonlinear solver

We can now load the JAX Grad-Shafranov static solver. The equilibrium is used to inform the solver of the computational domain and of the Green's functions of the coils. The limiter and profile objects become attributes of the solver. The JAX solver has two optional arguments:
- linear_solver: this determines the linear solver used to solve the linear GS problem. The options are:
	- 'dst' (default): A custom FFT-based solver that imposes boundary conditions using a discrete sine transform. This is  fast, scalable, and memory-efficient and is the recommended option for both CPU and GPU. 
	- 'dense': The dense inverse matrix corresponding to the elliptic operator is calculated at initialisation, and stored - when solving the GS problem, the solution to the linear GS problem is simply given by a fast matrix vector dot-product. This is the fastest option on GPU (for small domain sizes) but the memory cost is very high. 
	- 'sparse': This uses a sparse linear solver that reduces the memory cost, but increases the computational time. Not really recommended.
- precompute_boundary_greens: (Default True) this determines whether the Green's functions for the boundary condition is precomputed and stored as a large 3-dimensional array or whether it is computed on the fly at each solution step. This additional computational cost can be significant on the CPU but much less on the GPU, even for large meshes.

Note: It's not necessary to instantiate a new solver when aiming to use it on new or different equilibria, as long as the integration domain, mesh grid, and tokamak are consistent across solves. 

In [ ]:
from freegsnke.jaxify import GSstaticsolver
jGS = GSstaticsolver.NKGSsolver(eq, jProfile, jLimiter)   

### Define the coil currents

As mentioned before, during a forward solve, we use fixed coil currents (as well as given profile functions/parameters) as inputs to solve for the equilibrium, just as in the default FreeGSNKE example.

The difference here is that the coil currents and profile parameters are defined as arrays and tuples, so that they can be traced by JAX - in this way, we can work out gradients using JAX's AD capabilities.

In [ ]:
# load the coil currents
import pickle
with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
    currents_dict = pickle.load(f)
    
# assign currents to the eq object
for key in currents_dict.keys():
    eq.tokamak.set_coil_current(coil_label=key, current_value=currents_dict[key])
    
jCurrVec=jnp.asarray(eq.tokamak.getCurrentsVec())
jProfilePars=jProfile.init_params

### The forward solve

The syntax of a forward solve is similar to that of the FreeGSNKE forward solve with a few differences. The following arguments need to be passed to the 'solve' method:

- an initial guess of psi,
- the tuple of profile parameters,
- the coil currents vector,
- the target relative tolerance.

The JAX solver has the option of using a Newton-Raphson method or a Newton-Krylov method - this can be set at the time of solving the equilibrium using the 'solver-type' parameter - options are 'newton','newton-krylov', and 'jvp-newton-krylov' (uses JAX AD gradient instead of finite-difference approximation). 

By default, it is set to 'newton-krylov' and a Newton-Krylov method similar to that of default FreeGSNKE is used. In this case, the residual of the GS problem is defined as 
$$ F = \psi_{p} +  (\Delta^*)^{-1} \left( \mu_0 R J_{\phi}(\psi_{p},\psi_{c}) \right)$$
and the NK-method finds the $\psi_p$ that sets $F=0$. Note that in this case, calculating the residual requires solving a large linear system problem - involving the $\Delta^*$ elliptic operator.

If solver_type='newton', a Newton-Raphson method is used and the AD capability of JAX is used to generate an exact dense Jacobian matrix at each step. In this case, the residual of the GS problem is defined as 
$$ F = \Delta^*\psi_{p} +  \mu_0 R J_{\phi}(\psi_{p},\psi_{c}) $$
Note that in this case, calculating the residual does not require solving a large linear system problem - the $\Delta^*$ elliptic operator can simply be applied without explicitly creating a matrix.

Given the coil flux $\psi_c$ is known from the machine definition, coil currents and Green's functions prior to solving, we only require an initial guess for the plasma flux $\psi_p$. This is generated automatically in FreeGSNKE and scaled automatically according to the size of the coil currents and/or plasma current. If a good initial guess is known, it can be provided to the solver in the equilbirium object above via the `psi` option. 

The NR method is significantly slower than the NK method on CPU as the Jacobian matrix is being assembled at each iteration. On the GPU, this additional cost is less significant. If use_newton=True, a user can use the lag_Jacobian option to set whether the Jacobian should be calculated in every iteration of the method or if the user is happy to use an older Jacobian to speed up the solve. The stopping criterion is defined as

$$ \frac{\text{max} | \psi^{(n+1)} - \psi^{(n)} |}{\text{max} \ \psi^{(n)} - \text{min} \ \psi^{(n)}} < \varepsilon, $$

where $n$ is the iteration number. 

In [ ]:
# call the solver
psi_jax = jGS.solve(init_psi=eq.psi(),
                    profilePars = jProfilePars,
                    currentvec = jCurrVec,
                    target_relative_tolerance = 1e-9,
                    #solver_type='newton', 'newton-krylov (default)', 'jvp-newton-krylov'
                    verbose=True, # print output
                    )

We can compare the resulting equilibrium with that produced by the default solver in FreeGSNKE.

In [ ]:
# initialise the profiles
from freegsnke.jtor_update import ConstrainPaxisIp
profiles = ConstrainPaxisIp(
    eq=eq,
    paxis=8e3,
    Ip=6e5,
    fvac=0.5,
    alpha_m=1.8,
    alpha_n=1.2
)

# initialise the solver
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)

# call the solver, solution will be in eq.psi()
GSStaticSolver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-8,
    verbose=True
    )

# Check difference in psi
print("Difference between Jax and default solver:",jnp.linalg.norm(psi_jax-eq.psi()))

# Plot the differences
fig1, ax1 = plt.subplots(1, 1, figsize=(5, 8), dpi=80)
plt.contour(eq.R,eq.Z,eq.psi(),20,colors='black')
plt.contourf(eq.R, eq.Z, np.abs((eq.psi()-psi_jax)),50,cmap='hot_r')
eq.tokamak.plot(axis=ax1, show=False)
plt.plot(eq.tokamak.wall.R, eq.tokamak.wall.Z, 'k', 3.0)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()
plt.colorbar(); plt.show()

### Alternative profile functions
We can also use alternative profile functions. Here, we show how the Lao profile can be used. In the JAX solver, the profile is an atttribute of the solver - if the profile is changed, a new solver instance has to be used. However, in the JAX solver, this can be done easily as shown below.

In [ ]:
from freegsnke.jaxify.jtor import JLao85

# first get Lao parameters
alpha, beta = profiles.Lao_parameters(4,4)

# New profile with Lao parametrization
jProfileLao = JLao85(Ip=6e5,fvac=0.5,alpha=alpha,beta=beta)

# Create a new solver with Lao profile - this uses Equinox
# to replace the old profile object with the new profile
# JAX objects are immutable so a new copy is made
import equinox as eqx
jGSLao = eqx.tree_at(lambda x:x.profile, jGS, jProfileLao)

We can then use it directly in a new solve with the same coil currents - note that we can now use the solution from the previous profile as an initial guess.

In [ ]:
# call solver with new profile object
psi_jax_Lao = jGS.solve(init_psi=psi_jax,
                    profilePars = jProfilePars,
                    currentvec = jCurrVec,
                    target_relative_tolerance = 1e-9,
                    verbose=True, # print output
                    )


# compare the resulting equilbria 
print("Difference between old profile and Lao profile:",
        jnp.linalg.norm(psi_jax-psi_jax_Lao))

# Plot the differences
fig1, ax1 = plt.subplots(1, 1, figsize=(5, 8), dpi=80)
plt.contour(eq.R,eq.Z,psi_jax,20,colors='black')
plt.contourf(eq.R, eq.Z, np.abs((psi_jax_Lao-psi_jax)),50,cmap='hot_r')
eq.tokamak.plot(axis=ax1, show=False)
plt.plot(eq.tokamak.wall.R, eq.tokamak.wall.Z, 'k', 3.0)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()
plt.colorbar(); plt.show()

### Forward solve: limiter plasma

Here we use the saved limiter plasma currents from the default FreeGSNKE inverse solver notebook. Note that since the new profile object is of the same type (ConstrainPaxisIp) as in the original solve - we do not need to create a new solver. Only the profile parameters are different and we can extract these and pass these on to the solver. 

In [ ]:
# initialise the profiles
jProfile = JConstrainPaxisIp(
    paxis=6e3,
    Ip=4e5,
    fvac=0.5,
    alpha_m=1.8,
    alpha_n=1.2
)  

# set the coil currents
import pickle
with open('data/simple_limited_currents_PaxisIp.pk', 'rb') as f:
    current_values = pickle.load(f)

for key in current_values.keys():
    eq.tokamak.set_coil_current(coil_label=key, current_value=current_values[key])

jCurrVec_Limited = jnp.asarray(eq.tokamak.getCurrentsVec())

# carry out the foward solve to find the equilibrium
jProfilePars = jProfile.init_params
psi_jax_Limited = jGS.solve(init_psi=psi_jax,
                    profilePars = jProfilePars,
                    currentvec = jCurrVec_Limited,
                    target_relative_tolerance = 1e-9,
                    verbose=True, # print output
                    )

# call the solver, solution will be in eq.psi()
profiles_Lim = ConstrainPaxisIp(
eq=eq,
paxis=6e3,
Ip=4e5,
fvac=0.5,
alpha_m=1.8,
alpha_n=1.2
)
GSStaticSolver.solve(
eq=eq,
profiles=profiles_Lim,
constrain=None,
target_relative_tolerance=1e-9,
verbose=True
)
# Check difference in psi
print("Difference between Jax and default solver:",jnp.linalg.norm(psi_jax_Limited-eq.psi()))
# Plot the differences
fig1, ax1 = plt.subplots(1, 1, figsize=(5, 8), dpi=80)
plt.contour(eq.R,eq.Z,eq.psi(),20,colors='black')
plt.contourf(eq.R, eq.Z, np.abs((eq.psi()-psi_jax_Limited)),50,cmap='hot_r')
eq.tokamak.plot(axis=ax1, show=False)
plt.plot(eq.tokamak.wall.R, eq.tokamak.wall.Z, 'k', 3.0)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()
plt.colorbar(); plt.show()